# 🏠 AI Real Estate Analytics — Exploratory Data Analysis

This notebook walks through the complete EDA workflow for the PropSense AI project.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_collection.data_loader import load_data

sns.set_theme(style='darkgrid')
%matplotlib inline

## 1. Load Dataset

In [ ]:
df = load_data()
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.describe()

## 2. Missing Values

In [ ]:
missing = df.isnull().sum()
print(missing[missing > 0])

## 3. Price Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['price']/1e6, bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Price Distribution (₹ Millions)')
axes[0].set_xlabel('Price (₹M)')

axes[1].hist(np.log1p(df['price']), bins=60, color='seagreen', edgecolor='white')
axes[1].set_title('Log-Price Distribution')
axes[1].set_xlabel('log(Price + 1)')
plt.tight_layout()
plt.show()

## 4. City Analysis

In [ ]:
city_stats = df.groupby('city')['price'].agg(['mean','median','count']).sort_values('median', ascending=False)
city_stats['mean_lakhs'] = (city_stats['mean']/100000).round(1)
city_stats['median_lakhs'] = (city_stats['median']/100000).round(1)
city_stats[['mean_lakhs','median_lakhs','count']]

## 5. Correlation Matrix

In [ ]:
numeric_df = df.select_dtypes(include=[np.number]).drop(columns=['price_per_sqft'], errors='ignore')
corr = numeric_df.corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn', center=0, linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 6. Feature Engineering Preview

In [ ]:
from src.feature_engineering.feature_engineer import FeatureEngineer
fe = FeatureEngineer()
df_fe = fe.transform(df)
print('New features:', [c for c in df_fe.columns if c not in df.columns])
df_fe[['price_per_sqft','amenities_score','value_score','property_age_group']].head(10)

## 7. Model Training Summary

In [ ]:
import json
from pathlib import Path

metrics_path = Path('../outputs/model_comparison.json')
if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
    metrics_df = pd.DataFrame(metrics).T
    display(metrics_df)
else:
    print('Run train_pipeline.py first to generate model metrics.')